# Traning masif_ppi_search NN

### Prerequisites
- Training dataset of protein-protein complexes (PDB files)
- List files defining training/validation/test splits

### Step 1.1: Data Preparation

Prepare raw data (surfaces, features):
```bash
sbatch data_prepare.slurm
```

**What this does:**
- Downloads PDB files
- Triangulates molecular surfaces
- Computes geometric and chemical features
- Generates precomputation data (polar coordinates, etc.)

In [ ]:
# Run data_prepare.slurm
!sbatch data_prepare.slurm

In [1]:
import os
repo_root = !git rev-parse --show-toplevel
repo_root = repo_root[0]
masif_ppi_search_dir = os.path.join(repo_root, "masif/data/masif_ppi_search")
os.chdir(masif_ppi_search_dir)


In [2]:
# TODO: Visualize training data

# 1. Read training list and testing list
with open("lists/training.txt", "r") as f:
    training_list = f.readlines()
training_list = [line.strip() for line in training_list]

with open("lists/testing.txt", "r") as f:
    testing_list = f.readlines()
testing_list = [line.strip() for line in testing_list]


# Function to visualize a model_id with py3Dmol
import py3Dmol
def visualize_model_id(model_id):
    pdb_id = model_id.split("_")[0]
    chain1 = model_id.split("_")[1]
    chain2 = model_id.split("_")[2]
    
    chain_1_pdb_file = f"data_preparation/01-benchmark_pdbs/{pdb_id}_{chain1}.pdb"
    chain_2_pdb_file = f"data_preparation/01-benchmark_pdbs/{pdb_id}_{chain2}.pdb"

    chain_1_pdb_str = open(chain_1_pdb_file, "r").read()
    chain_2_pdb_str = open(chain_2_pdb_file, "r").read()
    
    # View the structure
    view = py3Dmol.view(width=600, height=500)
    view.addModel(chain_1_pdb_str, "pdb")
    view.setStyle({"model": 0}, {"cartoon": {"color": "cyan"}})
    view.addModel(chain_2_pdb_str, "pdb")
    view.setStyle({"model": 1}, {"cartoon": {"color": "green"}})
    view.zoomTo()
    view.show()
    
# Example: visualize the first training model
model_idx = 0
model_id = training_list[model_idx]
visualize_model_id(model_id)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
model_idx += 1
model_id = training_list[model_idx]
print(model_id)
visualize_model_id(model_id)

### Step 1.2: Cache Training Data

Cache the training data for masif_ppi_search:
```bash
sbatch cache_nn.slurm  # Or run cache_nn.sh
```

**What this does:**
- Loads precomputed features for all training proteins
- Samples positive (binder) and negative (non-binder) pairs
- Saves cached data as `.npy` files

**Outputs:**
- `nn_models/sc05/cache/binder_*.npy` - Binder features
- `nn_models/sc05/cache/pos_*.npy` - Positive site features
- `nn_models/sc05/cache/neg_*.npy` - Negative site features
- `nn_models/sc05/cache/*_idx.npy` - Train/val/test indices

**Use the settings in `custom_params_mixed_5to1.py`**

Implemented the negative-sampling upgrade for `masif_ppi_search` with configurable mixed negatives and loss balancing.

Should sample more negatives than positives. 

**Use the refactored 3-step workflow for cache generation to prevent OOM kill**

In [ ]:
# Step 1 - Build a global catalog of all cached features
!sbatch build_cache_catalog.slurm

In [ ]:
# Step 2 - Generate cache shards
!sbatch --array=0-999 cache_nn_shards.slurm

In [ ]:
# Step 3 - Merge cache shards
!sbatch merge_cache_shards.slurm

In [22]:
# Summary of cached features (exact positive pair mapping from catalog)
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path(".")  # or Path("masif/data/masif_ppi_search") if running from repo root
CACHE_DIR = DATA_DIR / "nn_models/sc05/cache"
CATALOG_DIR = CACHE_DIR / "catalog"


def parse_name(name):
    tokens = str(name).split("_")
    if len(tokens) < 5:
        return None, None, None
    pid = tokens[-2]
    try:
        vix = int(tokens[-1])
    except ValueError:
        return None, None, None
    ppi_pair_id = "_".join(tokens[:-2])
    return ppi_pair_id, pid, vix


def split_for(i, train, val, test):
    if i in train:
        return "train"
    if i in val:
        return "val"
    if i in test:
        return "test"
    return "?"


def build_exact_partner_lookup(records):
    lookup = {}
    ambiguous = 0
    for rec in records:
        item = rec.item() if hasattr(rec, "item") else rec
        ppi_pair_id = item.get("ppi_pair_id")
        k1 = np.asarray(item.get("k1", []), dtype=int)
        k2 = np.asarray(item.get("k2", []), dtype=int)
        n = min(len(k1), len(k2))
        for idx in range(n):
            key = (ppi_pair_id, int(k1[idx]))
            val = int(k2[idx])
            if key in lookup and lookup[key] != val:
                ambiguous += 1
                continue
            lookup[key] = val
    return lookup, ambiguous


def build_cache_summary():
    pos_names = np.load(CACHE_DIR / "pos_names.npy", allow_pickle=True)
    neg_names = np.load(CACHE_DIR / "neg_names.npy", allow_pickle=True)

    pos_train = set(np.load(CACHE_DIR / "pos_training_idx.npy"))
    pos_val = set(np.load(CACHE_DIR / "pos_val_idx.npy"))
    pos_test = set(np.load(CACHE_DIR / "pos_test_idx.npy"))
    neg_train = set(np.load(CACHE_DIR / "neg_training_idx.npy"))
    neg_val = set(np.load(CACHE_DIR / "neg_val_idx.npy"))
    neg_test = set(np.load(CACHE_DIR / "neg_test_idx.npy"))

    records = np.load(CATALOG_DIR / "records.npy", allow_pickle=True)
    exact_partner_lookup, ambiguous = build_exact_partner_lookup(records)

    rows = []

    # Positive rows: exact binder-partner mapping from catalog k1->k2
    exact_count = 0
    missing_count = 0
    for i, binder_name in enumerate(pos_names):
        ppi_pair_id, pid, binder_vix = parse_name(binder_name)
        partner_pid = "p2"
        partner_vix = None
        partner_name = None
        quality = "parse_error"

        if ppi_pair_id is not None and pid == "p1" and binder_vix is not None:
            key = (ppi_pair_id, int(binder_vix))
            if key in exact_partner_lookup:
                partner_vix = int(exact_partner_lookup[key])
                partner_name = f"{ppi_pair_id}_{partner_pid}_{partner_vix}"
                quality = "exact_catalog"
                exact_count += 1
            else:
                quality = "missing_in_catalog"
                missing_count += 1

        rows.append(
            {
                "type": "pos",
                "index": i,
                "split": split_for(i, pos_train, pos_val, pos_test),
                "binder_name": str(binder_name),
                "partner_name": partner_name,
                "binder_pid": pid,
                "partner_pid": partner_pid,
                "binder_vix": binder_vix,
                "partner_vix": partner_vix,
                "mapping_quality": quality,
            }
        )

    # Negative rows: standalone (no exact binder-negative linkage in final merged cache)
    for i, neg_name in enumerate(neg_names):
        _, pid, vix = parse_name(neg_name)
        rows.append(
            {
                "type": "neg",
                "index": i,
                "split": split_for(i, neg_train, neg_val, neg_test),
                "binder_name": str(neg_name),
                "partner_name": None,
                "binder_pid": pid,
                "partner_pid": None,
                "binder_vix": vix,
                "partner_vix": None,
                "mapping_quality": "standalone_negative",
            }
        )

    df = pd.DataFrame(rows)
    summary = df.groupby(["type", "split"]).size().unstack(fill_value=0)
    print("Counts by type and split:")
    display(summary)

    total_pos = len(pos_names)
    mapped_pct = (100.0 * exact_count / total_pos) if total_pos > 0 else 0.0
    print(
        f"Positive exact mapping coverage: {exact_count}/{total_pos} ({mapped_pct:.2f}%). "
        f"Missing={missing_count}, Ambiguous catalog entries skipped={ambiguous}"
    )
    display(df[df["type"] == "pos"]["mapping_quality"].value_counts(dropna=False))
    return df


df_cache = build_cache_summary()
display(df_cache.head())
print(f"df_cache.shape: {df_cache.shape}")

Counts by type and split:


split,test,train,val
type,,,
neg,10909,56530,3645
pos,2182,11306,729


Positive exact mapping coverage: 14217/14217 (100.00%). Missing=0, Ambiguous catalog entries skipped=0


mapping_quality
exact_catalog    14217
Name: count, dtype: int64

,type,index,split,binder_name,partner_name,binder_pid,partner_pid,binder_vix,partner_vix,mapping_quality
0,pos,0,train,1A0G_A_B_p1_5610,1A0G_A_B_p2_113,p1,p2,5610,113.0,exact_catalog
1,pos,1,train,1A0G_A_B_p1_23,1A0G_A_B_p2_4750,p1,p2,23,4750.0,exact_catalog
2,pos,2,train,1A0G_A_B_p1_3486,1A0G_A_B_p2_4163,p1,p2,3486,4163.0,exact_catalog
3,pos,3,train,1A0G_A_B_p1_2956,1A0G_A_B_p2_1152,p1,p2,2956,1152.0,exact_catalog
4,pos,4,train,1A0G_A_B_p1_3261,1A0G_A_B_p2_623,p1,p2,3261,623.0,exact_catalog


df_cache.shape: (85301, 10)


In [23]:
df_cache["mapping_quality"].value_counts()

mapping_quality
standalone_negative    71084
exact_catalog          14217
Name: count, dtype: int64

In [39]:
ENTRY_INDEX = 400

In [41]:
# Cell: Visualize selected cached feature
import numpy as np
import py3Dmol
import pymesh
from pathlib import Path
from Bio.PDB import PDBParser

# --- Config: select entry to visualize ---
ENTRY_TYPE = "pos"   # "pos" or "neg"
ENTRY_INDEX += 1      # index into pos or neg array
SHOW_FULL_PATCH = True  # True = patch vertices, False = center only

# --- Paths (relative to masif_ppi_search data dir) ---
DATA_DIR = Path(".")
CACHE_DIR = DATA_DIR / "nn_models/sc05/cache"
PRECOMP_DIR = DATA_DIR / "data_preparation/04b-precomputation_12A/precomputation"
PLY_DIR = DATA_DIR / "data_preparation/01-benchmark_surfaces"
PDB_DIR = DATA_DIR / "data_preparation/01-benchmark_pdbs"

def parse_name(name):
    """Parse '<ppi_pair_id>_<p1|p2>_<vertex_idx>' into components."""
    tokens = str(name).split("_")
    if len(tokens) < 5:
        return None, None, None

    pid = tokens[-2]
    if pid not in {"p1", "p2"}:
        return None, None, None

    try:
        vix = int(tokens[-1])
    except ValueError:
        return None, None, None

    ppi_pair_id = "_".join(tokens[:-2])
    if len(ppi_pair_id.split("_")) < 3:
        return None, None, None

    return ppi_pair_id, pid, vix


def load_vertices_from_ply(ply_fn):
    """Load vertices robustly across different mesh libraries/environments."""
    if hasattr(pymesh, "load_mesh"):
        return np.asarray(pymesh.load_mesh(str(ply_fn)).vertices)

    meshio = getattr(pymesh, "meshio", None)
    if meshio is not None and hasattr(meshio, "load_mesh"):
        return np.asarray(meshio.load_mesh(str(ply_fn)).vertices)

    try:
        import trimesh
    except ImportError as exc:
        raise RuntimeError(
            "No compatible mesh loader found. Install trimesh or use a PyMesh build with load_mesh()."
        ) from exc

    mesh = trimesh.load_mesh(str(ply_fn), process=False)
    return np.asarray(mesh.vertices)


def get_patch_coords(ppi_pair_id, pid, vix, full_patch=False):
    """Get patch center (and optionally full patch) coordinates in original frame."""
    fields = ppi_pair_id.split("_")
    if len(fields) < 3:
        return None, None
    pdb_id, ch1, ch2 = fields[0], fields[1], fields[2]
    chain = ch1 if pid == "p1" else ch2
    ply_fn = PLY_DIR / f"{pdb_id}_{chain}.ply"
    if not ply_fn.exists():
        return None, None

    vertices = load_vertices_from_ply(ply_fn)
    center = vertices[vix]
    if not full_patch:
        return center, None

    list_idx_fn = PRECOMP_DIR / ppi_pair_id / f"{pid}_list_indices.npy"
    if not list_idx_fn.exists():
        return center, None
    neigh = np.load(list_idx_fn, allow_pickle=True)[vix]
    patch_coords = vertices[neigh]
    return center, patch_coords

def add_struct_to_py3dmol(structure, view=None):
    from Bio.PDB import PDBIO
    from io import StringIO
    io = PDBIO()
    io.set_structure(structure)
    buf = StringIO()
    io.save(buf)
    if view is None:
        view = py3Dmol.view(width=600, height=500)
    view.addModel(buf.getvalue(), "pdb")
    return view

def add_spheres(view, coords, color, radius=0.3):
    if coords is None:
        return
    for pt in np.atleast_2d(coords):
        view.addSphere({"center": {"x": float(pt[0]), "y": float(pt[1]), "z": float(pt[2])}, "radius": radius, "color": color})


def nearest_vertex_idx(vertices, xyz):
    d2 = np.sum((vertices - xyz) ** 2, axis=1)
    return int(np.argmin(d2))


def infer_partner_name_from_pos_index(pos_idx, binder_name):
    ppi_pair_id, pid, binder_vix = parse_name(binder_name)
    if ppi_pair_id is None or pid != "p1" or binder_vix is None:
        return None

    fields = ppi_pair_id.split("_")
    if len(fields) < 3:
        return None

    pdb_id, ch1, ch2 = fields[0], fields[1], fields[2]
    binder_ply = PLY_DIR / f"{pdb_id}_{ch1}.ply"
    partner_ply = PLY_DIR / f"{pdb_id}_{ch2}.ply"
    if not binder_ply.exists() or not partner_ply.exists():
        return None

    binder_vertices = load_vertices_from_ply(binder_ply)
    partner_vertices = load_vertices_from_ply(partner_ply)
    if binder_vix < 0 or binder_vix >= len(binder_vertices):
        return None

    binder_center = binder_vertices[binder_vix]
    partner_vix = nearest_vertex_idx(partner_vertices, binder_center)
    return f"{ppi_pair_id}_p2_{partner_vix}"


# --- Load and parse ---
if ENTRY_TYPE != "pos":
    print("ENTRY_TYPE='neg' does not have exact binder-partner linkage in final merged cache.")
    print("Set ENTRY_TYPE='pos' for exact pair-aware visualization.")
else:
    pos_names = np.load(CACHE_DIR / "pos_names.npy", allow_pickle=True)
    if ENTRY_INDEX >= len(pos_names):
        print(f"Index {ENTRY_INDEX} out of range (max {len(pos_names)-1})")
    else:
        binder_name = str(pos_names[ENTRY_INDEX])
        partner_name = infer_partner_name_from_pos_index(ENTRY_INDEX, binder_name)

        if partner_name is None:
            print(f"Could not recover partner patch for positive index {ENTRY_INDEX} ({binder_name})")
        else:
            binder_ppi, binder_pid, binder_vix = parse_name(binder_name)
            partner_ppi, partner_pid, partner_vix = parse_name(partner_name)

            binder_center, binder_patch = get_patch_coords(
                binder_ppi, binder_pid, binder_vix, full_patch=SHOW_FULL_PATCH
            )
            partner_center, partner_patch = get_patch_coords(
                partner_ppi, partner_pid, partner_vix, full_patch=SHOW_FULL_PATCH
            )

            if binder_center is None or partner_center is None:
                print(f"Could not load pair coordinates for {binder_name} / {partner_name}")
            else:
                pdb_id, ch1, ch2 = binder_ppi.split("_")[:3]
                binder_chain_pdb = PDB_DIR / f"{pdb_id}_{ch1}.pdb"
                partner_chain_pdb = PDB_DIR / f"{pdb_id}_{ch2}.pdb"

                parser = PDBParser(QUIET=True)
                view = py3Dmol.view(width=700, height=520)
                loaded_chain = False

                if binder_chain_pdb.exists():
                    struct = parser.get_structure(binder_chain_pdb.stem, str(binder_chain_pdb))
                    add_struct_to_py3dmol(struct, view)
                    view.setStyle({"model": 0}, {"cartoon": {"color": "cyan"}})
                    loaded_chain = True

                if partner_chain_pdb.exists():
                    struct = parser.get_structure(partner_chain_pdb.stem, str(partner_chain_pdb))
                    add_struct_to_py3dmol(struct, view)
                    model_id = 1 if binder_chain_pdb.exists() else 0
                    view.setStyle({"model": model_id}, {"cartoon": {"color": "lightgreen"}})
                    loaded_chain = True

                if not loaded_chain:
                    print(
                        f"No chain PDB files found for pair: {binder_chain_pdb.name}, {partner_chain_pdb.name}"
                    )
                else:
                    add_spheres(view, binder_patch if SHOW_FULL_PATCH else binder_center, "blue", radius=0.22)
                    add_spheres(view, partner_patch if SHOW_FULL_PATCH else partner_center, "green", radius=0.22)
                    view.zoomTo()
                    view.show()

                print(
                    f"pos #{ENTRY_INDEX} | split=... | binder={binder_name} | partner={partner_name} | "
                    f"binder_center={binder_center} | partner_center={partner_center}"
                )

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

pos #402 | split=... | binder=1A22_A_B_p1_2277 | partner=1A22_A_B_p2_770 | binder_center=[ 61.53919983  36.64730072 129.52000427] | partner_center=[ 61.54499817  36.27999878 129.83500671]


### Step 1.3: Train MaSIF-ppi-search NN

Train the descriptor generation network:
```bash
sbatch masif_ppi_search_train.slurm
```

**Training configuration:**
- Network: Geometric CNN with rotation equivariance
- Loss: Siamese contrastive loss (push/pull descriptors)
- Output: 80-dimensional descriptors per vertex
- Duration: ~24-48 hours (depends on dataset size)

**Outputs:**
- `nn_models/sc05/all_feat/model_data/model.*` - Trained weights

In [ ]:
!sbatch masif_ppi_search_train.slurm

### Step 1.4: Generate Descriptors with New Model

Run inference with the newly trained model:
```bash
sbatch compute_descriptors.slurm
```

**What this does:**
- Loads the trained MaSIF-ppi-search weights
- Runs inference on all proteins
- Generates 80D descriptors for each surface vertex

**Expected output per protein complex:**
```
descriptors/sc05/all_feat/PDBID_CHAIN1_CHAIN2/
├── p1_desc_straight.npy    # Chain 1 descriptors (normal)
├── p1_desc_flipped.npy      # Chain 1 descriptors (flipped for complementarity)
├── p2_desc_straight.npy     # Chain 2 descriptors (normal)
└── p2_desc_flipped.npy      # Chain 2 descriptors (flipped)
```

In [ ]:
!sbatch compute_descriptors.slurm

In [ ]:
# Check the descriptors
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path(".")  # or Path("masif/data/masif_ppi_search") if running from repo root
desc_dir = DATA_DIR / "descriptors/sc05/all_feat"


def load_descriptors(model_id):
    desc_fn = desc_dir / f"{model_id}/p1_desc_straight.npy"
    return np.load(desc_fn)

# Example: load descriptors for a specific model
model_id = "1CQI_A_B"
descriptors = load_descriptors(model_id)
print(descriptors.shape)  # Should print (n_vertices, 80)
